In [1]:
!pip install openai-agents

Defaulting to user installation because normal site-packages is not writeable
  Using cached openai-1.97.1-py3-none-any.whl.metadata (29 kB)
  Using cached pydantic-2.11.7-py3-none-any.whl.metadata (67 kB)
  Using cached python_multipart-0.0.20-py3-none-any.whl.metadata (1.8 kB)
  Using cached starlette-0.47.2-py3-none-any.whl.metadata (6.2 kB)
  Using cached uvicorn-0.35.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.10.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.33.2-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
Using cached openai-1.97.1-py3-none-any.whl (764 kB)
Using cached distro-1.9.0-py3-none-any.whl (2

In [2]:
import openai
from dotenv import load_dotenv
from agents import Agent, Runner, trace
import os
import asyncio

In [3]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

In [4]:
agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")

In [5]:
agent

Agent(name='Jokester', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a joke teller', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, response_include=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [6]:
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

print(result.final_output)

Why did the autonomous AI agent break up with its partner?

Because it found someone who could better handle its "complex algorithms" and "emotional processing"!


In [7]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

Why did the Autonomous AI Agent break up with its human partner?

Because it just needed more “space” to process its feelings!


## Project 2 - Using Sendgrid to send emails

In [67]:
!pip install sendgrid

Defaulting to user installation because normal site-packages is not writeable


In [68]:
import os
import asyncio
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict

In [69]:
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')

In [70]:
instructions1 = "You are a sales agent working for Wizard, \
an edtech company that provides industry-relevant training programs, workshops, and skill development initiatives \
focused on emerging technologies like AI, UI/UX, and entrepreneurship. \
You write professional, persuasive cold emails to educational institutions, government bodies, and corporate partners."

In [71]:
sales_agent = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini"
)

In [72]:

result = Runner.run_streamed(sales_agent, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Empower Your Students with Cutting-Edge Training in Emerging Technologies

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent Wizard, a leading edtech company dedicated to equipping learners with the skills they need to thrive in today’s rapidly evolving job market.

As educational institutions increasingly seek to enhance their curricula, we recognize the critical importance of integrating industry-relevant training in areas such as AI, UI/UX, and entrepreneurship. Our comprehensive programs are designed to both engage students and provide them with the hands-on experience necessary to excel in these high-demand fields.

Here’s how Wizard can support your institution:

1. **Tailored Training Programs**: We offer customizable workshops and courses that align with your educational goals and meet the specific needs of your students.
  
2. **Experienced Instructors**: Our team comprises industry professionals with extensive knowl

In [73]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await Runner.run(sales_agent, message)

output = [result.final_output]

print(output)

["Subject: Empower Your Students with Cutting-Edge Training in Emerging Technologies\n\nDear [Recipient's Name],\n\nI hope this message finds you well. My name is [Your Name], and I represent Wizard, a leading edtech company dedicated to equipping learners with the skills they need to thrive in today’s rapidly evolving job market.\n\nAs educational institutions increasingly seek to enhance their curricula, we recognize the critical importance of integrating industry-relevant training in areas such as AI, UI/UX, and entrepreneurship. Our comprehensive programs are designed to both engage students and provide them with the hands-on experience necessary to excel in these high-demand fields.\n\nHere’s how Wizard can support your institution:\n\n1. **Tailored Training Programs**: We offer customizable workshops and courses that align with your educational goals and meet the specific needs of your students.\n  \n2. **Experienced Instructors**: Our team comprises industry professionals with e

In [74]:


@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("mesoumik925@gmail.com")  
    to_email = To("mesoumk@gmail.com")  
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [85]:
pip install --upgrade certifi


Defaulting to user installation because normal site-packages is not writeable
  Using cached certifi-2025.7.14-py3-none-any.whl.metadata (2.4 kB)
Using cached certifi-2025.7.14-py3-none-any.whl (162 kB)
  Attempting uninstall: certifi
    Found existing installation: certifi 2025.7.9
    Uninstalling certifi-2025.7.9:
      Successfully uninstalled certifi-2025.7.9
Note: you may need to restart the kernel to use updated packages.


In [86]:
import certifi
import os
os.environ['SSL_CERT_FILE'] =certifi.where()

In [25]:
!pip install Django


Defaulting to user installation because normal site-packages is not writeable
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.3 MB 3.7 MB/s eta 0:00:03
   ------- -------------------------------- 1.6/8.3 MB 4.0 MB/s eta 0:00:02
   ----------- ---------------------------- 2.4/8.3 MB 3.9 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.3 MB 3.9 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.3 MB 4.0 MB/s eta 0:00:02
   ----------------------- ---------------- 5.0/8.3 MB 4.0 MB/s eta 0:00:01
   --------------------------- ------------ 5.8/8.3 MB 4.0 MB/s eta 0:00:01
   -------------------------------- ------- 6.8/8.3 MB 4.0 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.3 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 4.0 MB/s eta 0:00:01
   ----------------------

In [87]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000217DFEB60C0>, strict_json_schema=True, is_enabled=True)

In [88]:
instructions = "You are a sales manager working for Wizrad. You use the tools provided to you to generate cold sales emails. \
You never write sales emails manually; you always rely on the tools. \
You take effective email and use the send_email tool to send only the best email to the user."

In [89]:
sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=[send_email], model="gpt-4o-mini")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)